## Settings

In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import pandas as pd
import numpy as np
import random
import os
import sys
import logging
import torch
import itertools

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [8]:
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)
project_root

'/Users/meina/Github/meina-t/matching_with_dl'

In [9]:
from logging import getLogger
logger = getLogger(__name__)
logger.setLevel(logging.INFO)

sv 
https://arxiv.org/pdf/2107.03427


In [10]:
from model import MatchingNet, train_model
from params import HParams
from data import Data
from loss.strategy_proofness import compute_spv
from loss.stability import compute_sv
from loss.efficiency import  compute_ev
from utils import is_pareto_dominates, da_with_t

## 準備

In [11]:
p_list = list(itertools.permutations([0.5, 0.3333, 0.1666]))
q_list = p_list + [(0.2,0.4,0.4), (0.4,0.2,0.4), (0.4,0.4,0.2),(0.25,.25,0.5), (0.25,0.5,0.25), (0.5,0.25,0.25), (0.3333,0.3333,0.3333)]

In [12]:
pairs_1_1 = []
pairs_1_2 = []
pairs_1_3 = []
pairs_2_1 = []
pairs_2_2 = []
pairs_2_3 = []
pairs_3_1 = []
pairs_3_2 = []
pairs_3_3 = []

p_set_1 = []
p_set_2 = []
p_set_3 = []
q_set_1 = []
q_set_2 = []
q_set_3 = []

for i in range(6):
    p_1 = p_list[i]
    p_set_1.append([p_1 for _ in range(3)])
for i in range(13):
    q_1 = q_list[i]
    q_set_1.append([q_1 for _ in range(3)])

# p_listから重複なく2つ選び、p_2を作成
for i in range(5):
    for j in range(i+1, 6):
        p_set_2.append([p_list[i], p_list[j], p_list[j]])

for i in range(12):
    for j in range(i+1, 13):
        q_set_2.append([q_list[i], q_list[j], q_list[j]])

for i in range(4):
    for j in range(i+1, 5):
        for k in range(j+1, 6):
            p_set_3.append([p_list[i], p_list[j], p_list[k]])

for i in range(11):
    for j in range(i+1, 12):
        for k in range(j+1, 13):
            q_set_3.append([q_list[i], q_list[j], q_list[k]])



for i in p_set_1:
    for j in q_set_1:
        pairs_1_1.append((i, j))

for i in p_set_1:
    for j in q_set_2:
        pairs_1_2.append((i, j))

for i in p_set_1:
    for j in q_set_3:
        pairs_1_3.append((i, j))

for i in p_set_2:
    for j in q_set_1:
        pairs_2_1.append((i, j))

for i in p_set_2:
    for j in q_set_2:
        pairs_2_2.append((i, j))

for i in p_set_2:
    for j in q_set_3:
        pairs_2_3.append((i, j))

for i in p_set_3:
    for j in q_set_1:
        pairs_3_1.append((i, j))

for i in p_set_3:
    for j in q_set_2:
        pairs_3_2.append((i, j))

for i in p_set_3:
    for j in q_set_3:
        pairs_3_3.append((i, j))

In [13]:
p_list = []
q_list = []

for pair in pairs_1_1:
    p_list.append(pair[0])
    q_list.append(pair[1])

## Model reconstruction

In [14]:
lambda_weights = torch.tensor([[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], dtype=torch.float32).to('mps') #重みを変更
model_path = 'model_15_03_aug_100000.pth' #モデルのパスを変更

In [16]:
device = 'mps'
cfg = HParams(
    num_agents=3,
    num_hidden_nodes=64,
    batch_size=256,
    epochs=100000,
    corr = 0,
    device='mps',
    lr=0.0001, # 違う場合には修正
    lambda_weights=lambda_weights,
    )

model = MatchingNet(cfg)
data = Data(cfg)

In [17]:
model.load_state_dict(torch.load(model_path))
model.to(device)
model.eval()

/var/folders/3s/sh8zstl54dl_t2y55d7th57m0000gn/T/ipykernel_22576/423538569.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


MatchingNet(
  (layers): Sequential(
    (0): Linear(in_features=18, out_features=64, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=64, out_features=64, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=64, out_features=64, bias=True)
    (7): LeakyReLU(negative_slope=0.01)
    (8): Linear(in_features=64, out_features=9, bias=True)
  )
)

## Store

In [18]:
def compute_matching(pairs):
    p_list = []
    q_list = []

    for pair in pairs:
        p_list.append(pair[0])
        q_list.append(pair[1])

    df_1_1 = pd.DataFrame({'p': p_list, 'q': q_list})

    matching_list = []
    spv_list = []
    sv_list = []
    ev_list = []
    da_matching_list = []
    da_spv_list = []
    da_sv_list = []
    da_ev_list = []

    for index, row in df_1_1.iterrows():
        p = torch.tensor(row['p'], dtype=torch.float32, device = 'mps') # torch.tensorに変換
        q = torch.tensor(row['q'], dtype=torch.float32, device = 'mps') # torch.tensorに変換
        p = p.unsqueeze(0) 
        q = q.unsqueeze(0) 

        matching = model(p, q) 
        spv = compute_spv(cfg, model, matching, p, q)
        sv = compute_sv(cfg,matching, p, q)
        ev = compute_ev(cfg, matching, p, q)

        da_matching = da_with_t(p, q)
        da_spv = compute_spv(cfg, da_with_t, da_matching, p, q)
        da_sv = compute_sv(cfg, da_matching, p, q)
        da_ev = compute_ev(cfg, da_matching, p, q)
        
        matching_list.append((torch.round(matching * 100) / 100).cpu().squeeze(0).tolist()) 
        spv_list.append(spv.sum().cpu().item())
        sv_list.append(sv.sum().cpu().item())
        ev_list.append(ev.sum().cpu().item())

        da_matching_list.append(da_matching.cpu().squeeze(0).tolist())
        da_spv_list.append(da_spv.sum().cpu().item())
        da_sv_list.append(da_sv.sum().cpu().item())
        da_ev_list.append(da_ev.sum().cpu().item())

    df_1_1['matching'] = matching_list
    df_1_1['matching'] = df_1_1['matching'].apply(lambda x: np.round(np.array(x), 2).tolist())
    df_1_1['spv'] = spv_list
    df_1_1['sv'] = sv_list
    df_1_1['ev'] = ev_list
    df_1_1['da_matching'] = da_matching_list
    df_1_1['da_spv'] = da_spv_list
    df_1_1['da_sv'] = da_sv_list
    df_1_1['da_ev'] = da_ev_list

    return df_1_1

In [19]:
df_1_1 = compute_matching(pairs_1_1)
print('finish 1_1')
df_1_2 = compute_matching(pairs_1_2)
print('finish 1_2')
df_1_3 = compute_matching(pairs_1_3)
print('finish 1_3')
df_2_1 = compute_matching(pairs_2_1)
print('finish 2_1')
df_2_2 = compute_matching(pairs_2_2)
print('finish 2_2')
df_2_3 = compute_matching(pairs_2_3)
print('finish 2_3')
df_3_1 = compute_matching(pairs_3_1)
print('finish 3_1')
df_3_2 = compute_matching(pairs_3_2)
print('finish 3_2')
df_3_3 = compute_matching(pairs_3_3)
print('finish 3_3')


finish 1_1
finish 1_2
finish 1_3
finish 2_1
finish 2_2
finish 2_3
finish 3_1
finish 3_2
finish 3_3


In [20]:
excel_file = 'output_15_03_aug_100000.xlsx' 

# Round all numerical columns to 2 decimal places
df_1_1 = df_1_1.round(2)
df_1_2 = df_1_2.round(2)
df_1_3 = df_1_3.round(2)
df_2_1 = df_2_1.round(2)
df_2_2 = df_2_2.round(2)
df_2_3 = df_2_3.round(2)
df_3_1 = df_3_1.round(2)
df_3_2 = df_3_2.round(2)
df_3_3 = df_3_3.round(2)

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    df_1_1.to_excel(writer, sheet_name='Sheet1_1', index=False)
    df_1_2.to_excel(writer, sheet_name='Sheet1_2', index=False)
    df_1_3.to_excel(writer, sheet_name='Sheet1_3', index=False)
    df_2_1.to_excel(writer, sheet_name='Sheet2_1', index=False)
    df_2_2.to_excel(writer, sheet_name='Sheet2_2', index=False)
    df_2_3.to_excel(writer, sheet_name='Sheet2_3', index=False)
    df_3_1.to_excel(writer, sheet_name='Sheet3_1', index=False)
    df_3_2.to_excel(writer, sheet_name='Sheet3_2', index=False)
    df_3_3.to_excel(writer, sheet_name='Sheet3_3', index=False)


In [32]:
df_1_1

,p,q,matching,spv,sv,ev,da_matching,da_spv,da_sv,da_ev
0,"[(0.5, 0.3333, 0.1666), (0.5, 0.3333, 0.1666),...","[(0.5, 0.3333, 0.1666), (0.5, 0.3333, 0.1666),...","[[0.44, 0.21, 0.35], [0.34, 0.4, 0.25], [0.21,...",0.12,0.83,-6.0,"[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, ...",0.0,0.0,-6.0
1,"[(0.5, 0.3333, 0.1666), (0.5, 0.3333, 0.1666),...","[(0.5, 0.1666, 0.3333), (0.5, 0.1666, 0.3333),...","[[0.29, 0.38, 0.34], [0.36, 0.25, 0.38], [0.35...",0.12,1.03,-6.0,"[[1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, 1.0, ...",0.0,0.0,-6.0
2,"[(0.5, 0.3333, 0.1666), (0.5, 0.3333, 0.1666),...","[(0.3333, 0.5, 0.1666), (0.3333, 0.5, 0.1666),...","[[0.4, 0.24, 0.36], [0.4, 0.3, 0.3], [0.2, 0.4...",0.15,0.84,-6.0,"[[0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, ...",0.0,0.0,-6.0
3,"[(0.5, 0.3333, 0.1666), (0.5, 0.3333, 0.1666),...","[(0.3333, 0.1666, 0.5), (0.3333, 0.1666, 0.5),...","[[0.13, 0.62, 0.25], [0.32, 0.16, 0.52], [0.55...",0.10,0.69,-6.0,"[[0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [1.0, 0.0, ...",0.0,0.0,-6.0
4,"[(0.5, 0.3333, 0.1666), (0.5, 0.3333, 0.1666),...","[(0.1666, 0.5, 0.3333), (0.1666, 0.5, 0.3333),...","[[0.27, 0.22, 0.51], [0.27, 0.54, 0.18], [0.46...",0.14,0.83,-6.0,"[[0.0, 0.0, 1.0], [1.0, 0.0, 0.0], [0.0, 1.0, ...",0.0,0.0,-6.0
...,...,...,...,...,...,...,...,...,...,...
73,"[(0.1666, 0.3333, 0.5), (0.1666, 0.3333, 0.5),...","[(0.4, 0.4, 0.2), (0.4, 0.4, 0.2), (0.4, 0.4, ...","[[0.24, 0.43, 0.33], [0.3, 0.3, 0.4], [0.46, 0...",0.10,0.49,-6.0,"[[0.0, 0.5, 0.5], [0.0, 0.5, 0.5], [1.0, 0.0, ...",0.0,0.0,-6.0
74,"[(0.1666, 0.3333, 0.5), (0.1666, 0.3333, 0.5),...","[(0.25, 0.25, 0.5), (0.25, 0.25, 0.5), (0.25, ...","[[0.52, 0.24, 0.24], [0.39, 0.21, 0.4], [0.09,...",0.09,0.45,-6.0,"[[0.5, 0.5, 0.0], [0.5, 0.5, 0.0], [0.0, 0.0, ...",0.0,0.0,-6.0
75,"[(0.1666, 0.3333, 0.5), (0.1666, 0.3333, 0.5),...","[(0.25, 0.5, 0.25), (0.25, 0.5, 0.25), (0.25, ...","[[0.18, 0.51, 0.31], [0.3, 0.24, 0.46], [0.52,...",0.21,0.52,-6.0,"[[0.5, 0.5, 0.0], [0.0, 0.0, 1.0], [0.5, 0.5, ...",0.0,0.0,-6.0
76,"[(0.1666, 0.3333, 0.5), (0.1666, 0.3333, 0.5),...","[(0.5, 0.25, 0.25), (0.5, 0.25, 0.25), (0.5, 0...","[[0.26, 0.24, 0.49], [0.24, 0.56, 0.2], [0.5, ...",0.05,0.45,-6.0,"[[0.0, 0.0, 1.0], [0.5, 0.5, 0.0], [0.5, 0.5, ...",0.0,0.0,-6.0


## add non-weighted spv

In [59]:
# read excel file
excel_file = 'output_[[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]].xlsx'

df_1_1 = pd.read_excel(excel_file, sheet_name='Sheet1_1')
df_1_2 = pd.read_excel(excel_file, sheet_name='Sheet1_2')
df_1_3 = pd.read_excel(excel_file, sheet_name='Sheet1_3')
df_2_1 = pd.read_excel(excel_file, sheet_name='Sheet2_1')
df_2_2 = pd.read_excel(excel_file, sheet_name='Sheet2_2')
df_2_3 = pd.read_excel(excel_file, sheet_name='Sheet2_3')
df_3_1 = pd.read_excel(excel_file, sheet_name='Sheet3_1')
df_3_2 = pd.read_excel(excel_file, sheet_name='Sheet3_2')
df_3_3 = pd.read_excel(excel_file, sheet_name='Sheet3_3')

In [60]:
df_list = [df_1_1, df_1_2, df_1_3, df_2_1, df_2_2, df_2_3, df_3_1, df_3_2, df_3_3]

In [61]:
def add_modeldata(df):
    ev_non_weighted_list = []
    da_ev_non_weighted_list = []

    for index, row in df.iterrows():
        p = torch.tensor(row['p'], dtype=torch.float32, device = 'mps') # torch.tensorに変換
        q = torch.tensor(row['q'], dtype=torch.float32, device = 'mps') # torch.tensorに変換
        p = p.unsqueeze(0) 
        q = q.unsqueeze(0) 

        matching = model(p, q) 
        ev_non_weighted = compute_ev(cfg, matching, p, q, non_weighted=True)
        ev_non_weighted_list.append(ev_non_weighted.sum().cpu().item())

        da_matching = da_with_t(p, q)
        da_ev_non_weighted = compute_ev(cfg, da_matching, p, q, non_weighted=True)
        da_ev_non_weighted_list.append(da_ev_non_weighted.sum().cpu().item())

    df['ev_non_weighted'] = ev_non_weighted_list
    df['da_ev_non_weighted'] = da_ev_non_weighted_list
    return df


In [62]:
for df in df_list:
    df['p'] = df['p'].apply(eval)
    df['q'] = df['q'].apply(eval)
    df = add_modeldata(df)


## ここまで

In [15]:
df_3_3[df_3_3['da_spv'] > 0.1]

,p,q,matching,spv,sv,ev,da_matching,da_spv,da_sv,da_ev


In [17]:
df_3_3[df_3_3['da_spv'] > 0]['p'].values[0]

[(0.5, 0.3333, 0.1666), (0.5, 0.1666, 0.3333), (0.3333, 0.1666, 0.5)]

In [18]:
df_3_3[df_3_3['da_spv'] > 0]['q'].values[0]

[(0.5, 0.1666, 0.3333), (0.3333, 0.5, 0.1666), (0.3333, 0.3333, 0.3333)]

In [19]:
df_3_3[df_3_3['da_spv'] > 0]['da_matching'].values[0]

[[1.0, 0.0, 0.0], [0.0, 0.5, 0.5], [0.0, 0.5, 0.5]]

In [29]:
p = torch.tensor([[(0.5, 0.3333, 0.1666), 
                   (0.5, 0.1666, 0.3333), 
                   (0.3333, 0.1666, 0.5)]], dtype=torch.float32).to(device)
q = torch.tensor([[(0.5, 0.1666, 0.3333), (0.3333, 0.5, 0.1666), (0.3333, 0.3333, 0.3333)]],  dtype=torch.float32).to(device)

In [25]:
da_matching = da_with_t(p, q)

In [26]:
da_matching

tensor([[[1.0000, 0.0000, 0.0000],
         [0.0000, 0.5000, 0.5000],
         [0.0000, 0.5000, 0.5000]]], device='mps:0')

In [27]:
compute_spv(cfg, da_with_t, da_matching, p, q)

SPV detected for agent 0 and f 2
SPV value: 8.940696716308594e-08
Misreported preferences (P_mis): tensor([[[[0.1667, 0.3333, 0.5000],
          [0.5000, 0.1666, 0.3333],
          [0.3333, 0.1666, 0.5000]],

         [[0.1667, 0.5000, 0.3333],
          [0.5000, 0.1666, 0.3333],
          [0.3333, 0.1666, 0.5000]],

         [[0.3333, 0.1667, 0.5000],
          [0.5000, 0.1666, 0.3333],
          [0.3333, 0.1666, 0.5000]],

         [[0.3333, 0.5000, 0.1667],
          [0.5000, 0.1666, 0.3333],
          [0.3333, 0.1666, 0.5000]],

         [[0.5000, 0.1667, 0.3333],
          [0.5000, 0.1666, 0.3333],
          [0.3333, 0.1666, 0.5000]],

         [[0.5000, 0.3333, 0.1667],
          [0.5000, 0.1666, 0.3333],
          [0.3333, 0.1666, 0.5000]]]], device='mps:0')
Individual values: tensor([[5.9605e-08, 0.0000e+00, 2.9802e-08, 0.0000e+00, 0.0000e+00, 0.0000e+00]],
       device='mps:0')


tensor([[0.0000e+00, 0.0000e+00, 8.9407e-08],
        [0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00]], device='mps:0')

In [28]:
p = torch.tensor([[[0.1667, 0.3333, 0.5000],
          [0.5000, 0.1666, 0.3333],
          [0.3333, 0.1666, 0.5000]]], dtype=torch.float32).to(device)
q = torch.tensor([[(0.5, 0.3333, 0.1666), 
                   (0.5, 0.1666, 0.3333), 
                   (0.3333, 0.3333, 0.3333)]],  dtype=torch.float32).to(device)

In [29]:
da_matching = da_with_t(p, q)
da_matching

tensor([[[0.0000, 0.5000, 0.5000],
         [1.0000, 0.0000, 0.0000],
         [0.0000, 0.5000, 0.5000]]], device='mps:0')

In [61]:
model = MatchingNet(cfg)
model.load_state_dict(torch.load('model_0120.pth'))
model.to(device)
model.eval()

/var/folders/3s/sh8zstl54dl_t2y55d7th57m0000gn/T/ipykernel_34642/395456298.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('model_0120.p

MatchingNet(
  (layers): Sequential(
    (0): Linear(in_features=18, out_features=64, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=64, out_features=64, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=64, out_features=64, bias=True)
    (7): LeakyReLU(negative_slope=0.01)
    (8): Linear(in_features=64, out_features=9, bias=True)
  )
)

In [67]:
df_1_1 = add_modeldata(df_1_1)
df_1_2 = add_modeldata(df_1_2)
df_1_3 = add_modeldata(df_1_3)
df_2_1 = add_modeldata(df_2_1)
df_2_2 = add_modeldata(df_2_2)
df_2_3 = add_modeldata(df_2_3)
df_3_1 = add_modeldata(df_3_1)
df_3_2 = add_modeldata(df_3_2)
df_3_3 = add_modeldata(df_3_3)


In [62]:
model(p, q)

tensor([[[2.0241e-01, 5.7346e-04, 7.9702e-01],
         [7.1915e-01, 2.4288e-01, 3.7965e-02],
         [7.8577e-02, 7.5729e-01, 1.6414e-01]]], device='mps:0',
       grad_fn=<DivBackward0>)

In [67]:
q_batch = q[0].cpu().numpy()
results = []
for i in range(3):  # 各行に対して処理
    row = q_batch[i].copy()
    unique_values, counts = np.unique(row, return_counts=True)
    
    for value, count in zip(unique_values, counts):
        if count == 1:
            continue

        elif count == 2:
            value_index = np.where(row == value)[0]
            for j in range(2):
                new_row = row.copy()
                new_row[value_index[j]] += 0.1
                new_result = q_batch.copy()
                new_result[i] = new_row
                results.append(new_result)

        elif count == 3:
            for p in itertools.permutations([0.2,0.1,0.0]):
                new_row = row.copy()
                for k in range(3):
                    new_row[k] += p[k]
                new_result = q_batch.copy()
                new_result[i] = new_row
                results.append(new_result)
if results == []:
    results.append(q_batch)
results

[array([[0.5   , 0.3333, 0.1666],
        [0.5   , 0.1666, 0.3333],
        [0.5333, 0.4333, 0.3333]], dtype=float32),
 array([[0.5   , 0.3333, 0.1666],
        [0.5   , 0.1666, 0.3333],
        [0.5333, 0.3333, 0.4333]], dtype=float32),
 array([[0.5   , 0.3333, 0.1666],
        [0.5   , 0.1666, 0.3333],
        [0.4333, 0.5333, 0.3333]], dtype=float32),
 array([[0.5   , 0.3333, 0.1666],
        [0.5   , 0.1666, 0.3333],
        [0.4333, 0.3333, 0.5333]], dtype=float32),
 array([[0.5   , 0.3333, 0.1666],
        [0.5   , 0.1666, 0.3333],
        [0.3333, 0.5333, 0.4333]], dtype=float32),
 array([[0.5   , 0.3333, 0.1666],
        [0.5   , 0.1666, 0.3333],
        [0.3333, 0.4333, 0.5333]], dtype=float32)]